# Project 28: Complaint and Grievance Management

**Team No.:** 22  
**Team Members:** Akash Mahalik; Piyush Acharya; Dipti Ranjan Mahakud; Soumya Ranjan Das  
**Proposed Hybrid Model:** IndicBERT + Complaint-Department GAT  
**Dataset:** Consumer complaints dataset — https://www.kaggle.com/datasets/shashwatwork/consume-complaints-dataset-fo-nlp

Self-contained Google Colab workflow. Run cells top to bottom; outputs are generated from the downloaded data and are intentionally not pre-populated.

## 0. Setup — Environment, Imports, Reproducibility

In [ ]:
!pip -q install kagglehub transformers sentencepiece torch-geometric captum pennylane tqdm tabulate
import os, json, random, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda:0")
print('Device:', DEVICE)


### CONFIG

In [ ]:
CONFIG={
 'project_no':'28','project_name':'Complaint and Grievance Management','team_no':'22','team_members':'Akash Mahalik; Piyush Acharya; Dipti Ranjan Mahakud; Soumya Ranjan Das',
 'task_type':'classification','dataset_name':'Consumer complaints dataset','kaggle_dataset_slug':'shashwatwork/consume-complaints-dataset-fo-nlp',
 'proposed_model':'mBERT + Complaint-Department GAT','target_column':None,'id_columns':[],'time_column':None,
 'split_ratios':{'train':.70,'val':.15,'test':.15},'random_seed':SEED,
 'data_raw_dir':'data/28/raw','data_processed_dir':'data/28/processed','figures_dir':'data/28/figures','results_dir':'data/28/results','reports_dir':'data/28/reports',
 'batch_size':64,'epochs':20,'patience':4,'learning_rate':1e-3,'max_rows':120000
}
for key in ['data_raw_dir','data_processed_dir','figures_dir','results_dir','reports_dir']: Path(CONFIG[key]).mkdir(parents=True,exist_ok=True)
CONFIG


## 1. Dataset Download

In [ ]:
import kagglehub, shutil
cache_path=Path(kagglehub.dataset_download(CONFIG['kaggle_dataset_slug']))
raw_dir=Path(CONFIG['data_raw_dir'])
for src in cache_path.rglob('*'):
    if src.is_file():
        dst=raw_dir/src.relative_to(cache_path); dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists(): shutil.copy2(src,dst)
raw_files=[p for p in raw_dir.rglob('*') if p.is_file()]
assert raw_files, 'Dataset download produced no files.'
assert sum(p.stat().st_size for p in raw_files)>1024, 'Downloaded payload is unexpectedly small.'
print(f'Discovered {len(raw_files)} files; total bytes={sum(p.stat().st_size for p in raw_files):,}')


## 2. Load Raw Data

In [ ]:
csvs=sorted(Path(CONFIG['data_raw_dir']).rglob('*.csv'),key=lambda p:p.stat().st_size,reverse=True); assert csvs,'No CSV found.'; raw_file=next((p for p in csvs if 'complaint' in p.name.lower()),csvs[0]); df=pd.read_csv(raw_file)
text_candidates=['narrative','consumer_complaint_narrative','complaint','text']; target_candidates=['product','department','category','label']; text_col=next((c for c in df.columns if c.lower() in text_candidates),None); target=next((c for c in df.columns if c.lower() in target_candidates),None); assert text_col and target,f'Required text/target absent: {list(df.columns)}'; CONFIG['target_column']=target
df=df.dropna(subset=[text_col,target]).drop_duplicates(subset=[text_col]).reset_index(drop=True); counts=df[target].value_counts(); df=df[df[target].isin(counts[counts>=20].index)];
if len(df)>30000: df=df.groupby(target,group_keys=False).apply(lambda x:x.sample(min(len(x),max(20,30000//df[target].nunique())),random_state=SEED)).reset_index(drop=True)
print(raw_file,text_col,target,df.shape)


## 3. Exploratory Data Analysis (EDA) + Data Quality Memo

In [ ]:
print('Shape:',df.shape); display(df.head()); print(df.dtypes.value_counts()); print('Duplicates:',int(df.duplicated().sum()))
missing=df.isna().mean().sort_values(ascending=False)
memo=f"""# Data Quality Memo

- Rows: {len(df):,}; columns: {df.shape[1]}.
- Duplicate rows: {df.duplicated().sum():,}.
- Highest missing fraction: {missing.max():.3f}.
- Target: `{target}` with {df[target].nunique()} observed values.
- Preprocessors are fitted only on training data.
- Dataset-specific leakage controls and schema assertions are applied below.
"""
Path(os.path.join(CONFIG["reports_dir"], "data_quality_memo.md")).write_text(memo,encoding='utf-8'); print(memo)


## 4. Preprocessing & Feature Engineering

The following split cell creates the partitions first and fits all learned preprocessing artifacts on the training partition only.

## 5. Train / Validation / Test Split

In [ ]:
train_df,rest=train_test_split(df,train_size=.7,stratify=df[target],random_state=SEED); val_df,test_df=train_test_split(rest,train_size=.5,stratify=rest[target],random_state=SEED); assert set(train_df.index).isdisjoint(test_df.index)
label_encoder=LabelEncoder().fit(train_df[target].astype(str)); y_train=label_encoder.transform(train_df[target].astype(str)); y_val=label_encoder.transform(val_df[target].astype(str)); y_test=label_encoder.transform(test_df[target].astype(str)); n_classes=len(label_encoder.classes_); feature_names=['IndicBERT embedding dimension '+str(i) for i in range(768)]
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
graph_vectorizer=TfidfVectorizer(max_features=5000,min_df=2).fit(train_df[text_col].astype(str)); train_tfidf=graph_vectorizer.transform(train_df[text_col].astype(str)); centroids=np.vstack([np.asarray(train_tfidf[y_train==i].mean(0)).ravel() for i in range(n_classes)]); similarity=cosine_similarity(centroids); np.fill_diagonal(similarity,0); graph_edges=[]
for i in tqdm(range(n_classes), desc="Training", unit="epoch"):
 for j in np.argsort(similarity[i])[-min(4,n_classes-1):]: graph_edges.extend([(i,int(j)),(int(j),i)])
department_edge_index=torch.tensor(sorted(set(graph_edges)),dtype=torch.long).t().contiguous(); assert department_edge_index.shape[1]>0
Path(os.path.join(CONFIG["data_processed_dir"], "split_manifest.json")).write_text(json.dumps({'train_rows':len(train_df),'val_rows':len(val_df),'test_rows':len(test_df),'stratified':True,'graph_source':'training-only TF-IDF product centroids','graph_rule':'four strongest cosine-similarity neighbors per product'},indent=2))


## 6. PyTorch Dataset & DataLoader

In [ ]:
from transformers import AutoTokenizer
MODEL_ID='bert-base-multilingual-cased'; tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
class TextDataset(Dataset):
 def __init__(self,frame,y): self.text=frame[text_col].astype(str).tolist(); self.y=y
 def __len__(self): return len(self.y)
 def __getitem__(self,i):
  z=tokenizer(self.text[i],max_length=128,truncation=True,padding='max_length',return_tensors='pt'); return z['input_ids'][0],z['attention_mask'][0],torch.tensor(self.y[i],dtype=torch.long)
train_loader=DataLoader(TextDataset(train_df,y_train),batch_size=16,shuffle=True); val_loader=DataLoader(TextDataset(val_df,y_val),batch_size=16); test_loader=DataLoader(TextDataset(test_df,y_test),batch_size=16)


## 7. Model Definitions

In [ ]:
from transformers import AutoModel
from torch_geometric.nn import GATConv

class IndicBERTDepartmentGAT(nn.Module):

    def __init__(self, k):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_ID)
        h = self.bert.config.hidden_size
        self.department = nn.Parameter(torch.randn(k, h) * 0.02)
        self.gat1 = GATConv(h, h // 4, heads=4)
        self.gat2 = GATConv(h, h)
        self.head = nn.Linear(h * 2, k)
        self.register_buffer('edge_index', department_edge_index.clone())

    def parameter_groups(self):
        bert_ids = {id(p) for p in self.bert.parameters()}
        new = [p for p in self.parameters() if id(p) not in bert_ids]
        return [{'params': self.bert.parameters(), 'lr': 2e-05}, {'params': new, 'lr': 0.001}]

    def forward(self, ids, mask):
        text = self.bert(input_ids=ids, attention_mask=mask).last_hidden_state[:, 0]
        graph = torch.relu(self.gat1(self.department, self.edge_index))
        graph = self.gat2(graph, self.edge_index)
        context = torch.softmax(text @ graph.t() / np.sqrt(graph.shape[1]), 1) @ graph
        return self.head(torch.cat([text, context], 1))
hybrid = IndicBERTDepartmentGAT(n_classes)
loss_fn = lambda model, b: nn.functional.cross_entropy(model(b[0], b[1]), b[2])


## 8. Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, loss_fn, checkpoint, epochs=None):
    epochs = epochs or CONFIG['epochs']
    model = model.to(DEVICE)
    grouped = hasattr(model, 'parameter_groups')
    opt = torch.optim.AdamW(model.parameter_groups() if grouped else model.parameters(), lr=CONFIG['learning_rate'], weight_decay=0.0001)
    total_steps = max(1, epochs * len(train_loader))
    warmup_steps = max(1, int(0.1 * total_steps))
    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lambda step: min((step + 1) / warmup_steps, 1.0)) if grouped else torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=2, factor=0.5)
    best = float('inf')
    stale = 0
    history = {'train_loss': [], 'val_loss': []}
    for epoch in tqdm(range(epochs), desc='Training', unit='epoch'):
        model.train()
        total = 0
        for batch in train_loader:
            batch = [v.to(DEVICE) for v in batch]
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model, batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if grouped:
                scheduler.step()
            total += loss.item() * batch[0].shape[0]
        train_loss = total / len(train_loader.dataset)
        model.eval()
        total = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = [v.to(DEVICE) for v in batch]
                total += loss_fn(model, batch).item() * batch[0].shape[0]
        val_loss = total / len(val_loader.dataset)
        if not grouped:
            scheduler.step(val_loss)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        print(f'Epoch {epoch + 1:02d}: train={train_loss:.4f}, val={val_loss:.4f}')
        if val_loss < best - 1e-05:
            best = val_loss
            stale = 0
            torch.save(model.state_dict(), checkpoint)
        else:
            stale += 1
            if stale >= CONFIG['patience']:
                break
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
    return history
hybrid_history = train_model(hybrid, train_loader, val_loader, loss_fn, 'results/best_hybrid.pt', 8)


## 9. Evaluation Metrics

In [ ]:
def predict(model):
    model.eval()
    probs = []
    ys = []
    with torch.no_grad():
        for ids, mask, y in test_loader:
            probs.append(torch.softmax(model(ids.to(DEVICE), mask.to(DEVICE)), 1).cpu().numpy())
            ys.append(y.numpy())
    return (np.vstack(probs), np.concatenate(ys))

def metrics(prob, y):
    pred = prob.argmax(1)
    pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average='macro', zero_division=0)
    return ({'accuracy': float(accuracy_score(y, pred)), 'precision_macro': float(pr), 'recall_macro': float(rc), 'f1_macro': float(f1)}, pred)
hybrid_prob, test_y2 = predict(hybrid)
results = {}
results['hybrid'], hybrid_pred = metrics(hybrid_prob, y_test)
Path('results/metrics.json').write_text(json.dumps(results, indent=2))
print(results)


## 10. Required Figures

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(hybrid_history['train_loss'], label='Hybrid train')
plt.plot(hybrid_history['val_loss'], label='Hybrid val')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.tight_layout()
plt.savefig('figures/fig01_loss_curves.png', dpi=150)
plt.show()
cm = confusion_matrix(y_test, hybrid_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('figures/fig02_confusion_matrix.png', dpi=150)
plt.show()
plt.figure(figsize=(8, 5))
for k in tqdm(range(n_classes), desc='Training', unit='epoch'):
    binary = (y_test == k).astype(int)
    if binary.min() != binary.max():
        precision, recall, _ = precision_recall_curve(binary, hybrid_prob[:, k])
        plt.plot(recall, precision, label=str(label_encoder.classes_[k]))
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig('figures/fig03_precision_recall.png', dpi=150)
plt.show()
sample = next(iter(test_loader))
ids, mask, _ = sample
ids = ids[:32].to(DEVICE)
mask = mask[:32].to(DEVICE)
emb = hybrid.bert.get_input_embeddings()(ids).detach().requires_grad_(True)
out = hybrid.bert(inputs_embeds=emb, attention_mask=mask).last_hidden_state[:, 0]
logits = hybrid.head(torch.cat([out, out], 1))
logits.max(1).values.sum().backward()
importance = emb.grad.abs().mean((0, 2)).cpu().numpy()
plt.figure(figsize=(8, 5))
plt.plot(importance)
plt.xlabel('Token position')
plt.ylabel('Gradient importance')
plt.tight_layout()
plt.savefig('figures/fig04_feature_importance.png', dpi=150)
plt.show()
errors = (hybrid_pred != y_test).astype(int)
plt.figure(figsize=(8, 4))
pd.Series(errors).rolling(max(5, len(errors) // 40), min_periods=1).mean().plot()
plt.ylabel('Rolling error rate')
plt.tight_layout()
plt.savefig('figures/fig05_error_analysis.png', dpi=150)
plt.show()
metric_names = ['accuracy', 'f1_macro']
x = np.arange(2)
plt.figure(figsize=(6, 4))
plt.bar(x, [results['hybrid'][m] for m in metric_names], label='hybrid')
plt.xticks(x, metric_names)
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.savefig('figures/fig06_proposed_metrics.png', dpi=150)
plt.show()
